# Recycling Awareness — Exploratory Data Analysis
**MS Data Science & MS Computer Science — Final Semester Project**

This notebook performs a full EDA on three datasets:
- `clean_recycling_data.csv` — US national EPA data (1960–2018, 4 materials)
- `regional_data.csv` — 8 US states, material-level rates
- `global_data.csv` — 15 OECD countries, overall MSW recycling rates

**Contents**
1. Setup & Data Loading
2. Dataset Overview
3. National Trends Analysis
4. Statistical Summary
5. Regional Analysis
6. Global Comparison
7. Correlation Analysis
8. Key Findings & Insights

---
## 1. Setup & Data Loading

In [ ]:
# Install if needed — uncomment and run once
# !pip install pandas matplotlib seaborn scipy

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.facecolor': '#fafafa',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

COLORS = {
    'Plastic': '#ef5350',
    'Paper':   '#43a047',
    'Glass':   '#1e88e5',
    'Metal':   '#fb8c00'
}

print('✅ Libraries loaded successfully')

In [ ]:
from pathlib import Path

# Works whether notebook is in project root or a notebooks/ subfolder
here = Path().resolve()
data_dir = next(
    (p / 'data' / 'processed' for p in [here, here.parent, here.parent.parent]
     if (p / 'data' / 'processed').exists()),
    here  # fallback
)

national = pd.read_csv(data_dir / 'clean_recycling_data.csv')
regional = pd.read_csv(data_dir / 'regional_data.csv')
global_df = pd.read_csv(data_dir / 'global_data.csv')

print(f'National : {national.shape[0]} rows  |  Regional : {regional.shape[0]} rows  |  Global : {global_df.shape[0]} rows')
print(f'Data path: {data_dir}')

---
## 2. Dataset Overview

In [ ]:
print('=== NATIONAL DATASET ===')
print(national.head(8))
print(f'\nYears covered : {national["year"].min()} – {national["year"].max()}')
print(f'Materials     : {sorted(national["material"].unique())}')
print(f'Missing values: {national.isnull().sum().sum()}')

In [ ]:
print('=== REGIONAL DATASET ===')
print(regional.head(8))
print(f'\nStates  : {sorted(regional["region"].unique())}')
print(f'Missing : {regional.isnull().sum().sum()}')

In [ ]:
print('=== GLOBAL DATASET ===')
print(global_df.head(8))
print(f'\nCountries : {sorted(global_df["country"].unique())}')
print(f'Missing   : {global_df.isnull().sum().sum()}')

---
## 3. National Trends Analysis
### 3a. Recycling rates over time — all 4 materials

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for material, grp in national.groupby('material'):
    grp_sorted = grp.sort_values('year')
    ax.plot(
        grp_sorted['year'], grp_sorted['recycling_rate'],
        marker='o', markersize=5, linewidth=2.5,
        color=COLORS[material], label=material
    )
    # label the last point
    last = grp_sorted.iloc[-1]
    ax.annotate(
        f"{last['recycling_rate']:.1f}%",
        xy=(last['year'], last['recycling_rate']),
        xytext=(4, 0), textcoords='offset points',
        fontsize=9, color=COLORS[material], fontweight='bold'
    )

ax.set_title('US Recycling Rates by Material (1960–2018)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Recycling Rate (%)', fontsize=11)
ax.set_ylim(0, 80)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('chart_national_trends.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: chart_national_trends.png')

### 3b. Decade-over-decade growth

In [ ]:
# Pivot to wide format for easy comparison across decades
pivot = national.pivot_table(index='year', columns='material', values='recycling_rate')

# Year-over-year change (absolute percentage points)
pivot_change = pivot.diff().dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw rates
pivot.plot(ax=axes[0], color=[COLORS[c] for c in pivot.columns], marker='o', markersize=4)
axes[0].set_title('Recycling Rate (%) by Year', fontweight='bold')
axes[0].set_ylabel('Rate (%)')
axes[0].set_ylim(0, 80)

# Right: period-over-period change
pivot_change.plot(ax=axes[1], color=[COLORS[c] for c in pivot_change.columns],
                  marker='o', markersize=4)
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('Change vs Previous Data Point (pp)', fontweight='bold')
axes[1].set_ylabel('Percentage point change')

plt.suptitle('US Recycling — Rates and Change Over Time', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart_decade_change.png', bbox_inches='tight', dpi=150)
plt.show()

### 3c. Growth multiplier — how many times has each material improved?

In [ ]:
growth = {}
for material in national['material'].unique():
    sub  = national[national['material'] == material].sort_values('year')
    first = sub['recycling_rate'].iloc[0]
    last  = sub['recycling_rate'].iloc[-1]
    # avoid division by zero for early plastic data
    if first > 0:
        growth[material] = round(last / first, 1)
    else:
        growth[material] = float('inf')

print('Growth multiplier (2018 rate ÷ earliest rate):')
for m, g in sorted(growth.items(), key=lambda x: -x[1]):
    bar = '█' * min(int(g), 30)
    val = f'{g}×' if g != float('inf') else '∞ (started at 0%)'
    print(f'  {m:<8} {bar} {val}')

---
## 4. Statistical Summary
### 4a. Descriptive statistics per material

In [ ]:
stats_df = national.groupby('material')['recycling_rate'].agg(
    Count='count',
    Mean=lambda x: round(x.mean(), 2),
    Median=lambda x: round(x.median(), 2),
    Std=lambda x: round(x.std(), 2),
    Min=lambda x: round(x.min(), 2),
    Max=lambda x: round(x.max(), 2),
    Range=lambda x: round(x.max() - x.min(), 2)
).reset_index()

print('=== Descriptive Statistics — US National Data ===')
print(stats_df.to_string(index=False))

### 4b. Distribution of recycling rates per material

In [ ]:
materials = sorted(national['material'].unique())
fig, axes = plt.subplots(1, 4, figsize=(15, 4), sharey=False)

for ax, mat in zip(axes, materials):
    data = national[national['material'] == mat]['recycling_rate']
    ax.hist(data, bins=8, color=COLORS[mat], alpha=0.8, edgecolor='white')
    ax.axvline(data.mean(), color='black', linestyle='--', linewidth=1.2, label=f'Mean: {data.mean():.1f}%')
    ax.set_title(mat, fontweight='bold', color=COLORS[mat])
    ax.set_xlabel('Rate (%)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.suptitle('Distribution of Recycling Rates by Material (all years)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('chart_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

### 4c. Box plot — spread and outliers

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

data_by_material = [national[national['material'] == m]['recycling_rate'].values for m in materials]
bp = ax.boxplot(data_by_material, patch_artist=True, labels=materials,
                medianprops={'color': 'black', 'linewidth': 2})

for patch, mat in zip(bp['boxes'], materials):
    patch.set_facecolor(COLORS[mat])
    patch.set_alpha(0.7)

ax.set_title('Recycling Rate Distribution — Box Plot (1960–2018)', fontsize=13, fontweight='bold')
ax.set_ylabel('Recycling Rate (%)')
ax.set_xlabel('Material')
plt.tight_layout()
plt.savefig('chart_boxplot.png', bbox_inches='tight', dpi=150)
plt.show()

### 4d. Linear regression — trend significance test

In [ ]:
print('=== Linear Regression: Year → Recycling Rate (per material) ===\n')
print(f'{"Material":<10} {"Slope (pp/yr)":<16} {"R²":<10} {"p-value":<12} {"Trend"}')
print('-' * 62)

for mat in materials:
    sub = national[national['material'] == mat].sort_values('year')
    slope, intercept, r, p, se = stats.linregress(sub['year'], sub['recycling_rate'])
    trend = '↑ Significant' if p < 0.05 and slope > 0 else ('↓ Declining' if slope < 0 else 'No trend')
    print(f'{mat:<10} {slope:+.3f}{" ":<11} {r**2:.3f}{" ":<5} {p:.4f}{" ":<7} {trend}')

In [ ]:
# Visual: regression lines overlaid on scatter
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for ax, mat in zip(axes, materials):
    sub = national[national['material'] == mat].sort_values('year')
    slope, intercept, r, p, _ = stats.linregress(sub['year'], sub['recycling_rate'])
    
    ax.scatter(sub['year'], sub['recycling_rate'], color=COLORS[mat], s=60, zorder=3)
    xs = sub['year']
    ax.plot(xs, slope * xs + intercept, color='black', linewidth=1.5, linestyle='--', alpha=0.7)
    ax.set_title(f'{mat}  |  slope={slope:+.2f} pp/yr  |  R²={r**2:.2f}', fontweight='bold')
    ax.set_xlabel('Year'); ax.set_ylabel('Rate (%)')
    ax.set_ylim(0, max(sub['recycling_rate']) * 1.2)

plt.suptitle('Linear Trend per Material (dashed = regression line)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('chart_regression.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 5. Regional Analysis
### 5a. State comparison — all materials

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, mat in zip(axes, materials):
    sub = regional[regional['material'] == mat].sort_values('recycling_rate', ascending=True)
    bars = ax.barh(sub['region'], sub['recycling_rate'],
                   color=COLORS[mat], alpha=0.85, edgecolor='white')
    # add value labels
    for bar, val in zip(bars, sub['recycling_rate']):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=9)
    ax.set_title(f'{mat} Recycling Rate by State', fontweight='bold', color=COLORS[mat])
    ax.set_xlabel('Recycling Rate (%)')
    ax.set_xlim(0, 105)
    ax.axvline(sub['recycling_rate'].mean(), color='black', linestyle='--',
               linewidth=1, alpha=0.6, label=f'Avg: {sub["recycling_rate"].mean():.1f}%')
    ax.legend(fontsize=8)

plt.suptitle('Recycling Rates by US State and Material (2019–2020)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('chart_regional.png', bbox_inches='tight', dpi=150)
plt.show()

### 5b. Heatmap — state × material

In [ ]:
heat = regional.pivot_table(index='region', columns='material', values='recycling_rate')
heat = heat[['Plastic', 'Glass', 'Metal', 'Paper']]  # consistent order

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(
    heat, annot=True, fmt='.1f', cmap='YlGn',
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Recycling Rate (%)'},
    ax=ax
)
ax.set_title('Recycling Rate Heatmap — State × Material', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Material')
ax.set_ylabel('State')
plt.tight_layout()
plt.savefig('chart_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print('\nStates sorted by overall recycling performance:')
print(heat.mean(axis=1).sort_values(ascending=False).round(1).to_string())

### 5c. Deposit law states vs non-deposit states

In [ ]:
deposit_states     = ['Michigan', 'Oregon', 'New York']
no_deposit_states  = ['Texas', 'Florida', 'Virginia', 'California', 'Washington']

regional['policy'] = regional['region'].apply(
    lambda r: 'Deposit law' if r in deposit_states else 'No deposit'
)

policy_summary = regional.groupby(['policy', 'material'])['recycling_rate'].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(materials))
width = 0.35

deposit_vals    = [policy_summary[(policy_summary['policy']=='Deposit law') &
                                  (policy_summary['material']==m)]['recycling_rate'].values[0]
                   for m in materials]
no_deposit_vals = [policy_summary[(policy_summary['policy']=='No deposit') &
                                  (policy_summary['material']==m)]['recycling_rate'].values[0]
                   for m in materials]

bars1 = ax.bar([i - width/2 for i in x], deposit_vals,    width, label='Deposit law states',  color='#2e7d32', alpha=0.85)
bars2 = ax.bar([i + width/2 for i in x], no_deposit_vals, width, label='No deposit states', color='#ef9a9a', alpha=0.85)

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f'{bar.get_height():.1f}%', ha='center', fontsize=9)

ax.set_xticks(list(x)); ax.set_xticklabels(materials)
ax.set_ylabel('Average Recycling Rate (%)')
ax.set_title('Impact of Bottle Deposit Laws on Recycling Rates', fontsize=13, fontweight='bold')
ax.legend()
ax.set_ylim(0, 110)
plt.tight_layout()
plt.savefig('chart_deposit_law.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 6. Global Comparison

In [ ]:
# Most recent rate per country
latest_global = (
    global_df.sort_values('year')
    .groupby('country')
    .last()
    .reset_index()
    .sort_values('recycling_rate', ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 7))

colors = ['#ef5350' if c == 'United States' else '#2e7d32' for c in latest_global['country']]
bars = ax.barh(latest_global['country'], latest_global['recycling_rate'],
               color=colors, alpha=0.85, edgecolor='white')

for bar, val in zip(bars, latest_global['recycling_rate']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)

ax.set_title('Global MSW Recycling Rates — Most Recent Year', fontsize=13, fontweight='bold')
ax.set_xlabel('Recycling Rate (%) — All Municipal Solid Waste')
ax.set_xlim(0, 65)
ax.axvline(latest_global[latest_global['country']=='United States']['recycling_rate'].values[0],
           color='#ef5350', linestyle='--', linewidth=1.2, alpha=0.7, label='US rate')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('chart_global.png', bbox_inches='tight', dpi=150)
plt.show()

us_rate   = latest_global[latest_global['country']=='United States']['recycling_rate'].values[0]
us_rank   = (latest_global['recycling_rate'] < us_rate).sum() + 1
top_rate  = latest_global['recycling_rate'].max()
top_country = latest_global.iloc[-1]['country']
print(f'\nUS rate  : {us_rate}%')
print(f'US rank  : {us_rank} out of {len(latest_global)} countries shown')
print(f'Top rate : {top_country} at {top_rate}%')
print(f'Gap      : {top_rate - us_rate:.1f} percentage points behind the leader')

---
## 7. Correlation Analysis
### 7a. Are materials correlated? (do they improve together?)

In [ ]:
pivot_corr = national.pivot_table(index='year', columns='material', values='recycling_rate')
corr_matrix = pivot_corr.corr().round(2)

fig, ax = plt.subplots(figsize=(7, 5))
mask = pd.DataFrame(False, index=corr_matrix.index, columns=corr_matrix.columns)
import numpy as np
mask.values[np.triu_indices_from(mask.values, k=1)] = True

sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
    vmin=-1, vmax=1, center=0,
    linewidths=0.5, linecolor='white',
    mask=mask, ax=ax
)
ax.set_title('Correlation Between Material Recycling Rates\n(1960–2018)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('chart_correlation.png', bbox_inches='tight', dpi=150)
plt.show()

print('\nInterpretation:')
print('  Values close to +1.0 = materials improve together (driven by same policies/era)')
print('  Values close to 0    = materials are independent')

### 7b. Regional correlation — plastic vs glass rates across states

In [ ]:
reg_pivot = regional.pivot_table(index='region', columns='material', values='recycling_rate')

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(reg_pivot['Plastic'], reg_pivot['Glass'],
           color='#2e7d32', s=80, zorder=3)

for state in reg_pivot.index:
    ax.annotate(state,
                xy=(reg_pivot.loc[state, 'Plastic'], reg_pivot.loc[state, 'Glass']),
                xytext=(4, 4), textcoords='offset points', fontsize=8)

slope, intercept, r, p, _ = stats.linregress(reg_pivot['Plastic'], reg_pivot['Glass'])
xs = reg_pivot['Plastic']
ax.plot(xs, slope*xs + intercept, '--', color='gray', linewidth=1.2)

ax.set_xlabel('Plastic Recycling Rate (%)')
ax.set_ylabel('Glass Recycling Rate (%)')
ax.set_title(f'Plastic vs Glass Recycling Rate by State\n(r={r:.2f}, p={p:.3f})', fontweight='bold')
plt.tight_layout()
plt.savefig('chart_scatter_regional.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 8. Key Findings & Insights

In [ ]:
print('=' * 65)
print('  KEY FINDINGS — Recycling Awareness EDA')
print('=' * 65)

# Finding 1: Plastic
plastic = national[national['material']=='Plastic']
print(f'''
1. PLASTIC CRISIS
   Plastic recycling has NEVER exceeded 10% in the US.
   Range: {plastic["recycling_rate"].min():.1f}% – {plastic["recycling_rate"].max():.1f}%
   2018 rate: {plastic[plastic["year"]==2018]["recycling_rate"].values[0]:.1f}%
   Despite massive growth in plastic production, recycling barely moved.
''')

# Finding 2: Paper
paper = national[national['material']=='Paper']
p1960 = paper[paper['year']==1960]['recycling_rate'].values[0]
p2018 = paper[paper['year']==2018]['recycling_rate'].values[0]
print(f'''2. PAPER SUCCESS STORY
   Paper went from {p1960:.1f}% in 1960 to {p2018:.1f}% in 2018 (+{p2018-p1960:.1f} pp).
   Strongest upward trend of all materials (R² highest).
   Driven by curbside programs and industry mandates.
''')

# Finding 3: Deposit law impact
deposit_glass    = regional[regional['region'].isin(deposit_states) & (regional['material']=='Glass')]['recycling_rate'].mean()
no_deposit_glass = regional[regional['region'].isin(no_deposit_states) & (regional['material']=='Glass')]['recycling_rate'].mean()
print(f'''3. BOTTLE DEPOSIT LAWS — MASSIVE IMPACT
   Average glass rate in DEPOSIT states     : {deposit_glass:.1f}%
   Average glass rate in NON-DEPOSIT states : {no_deposit_glass:.1f}%
   Deposit states recycle {deposit_glass/no_deposit_glass:.1f}× more glass.
   Michigan alone reaches {regional[(regional["region"]=="Michigan") & (regional["material"]=="Glass")]["recycling_rate"].values[0]:.1f}% (10-cent deposit).
''')

# Finding 4: Best/worst state
state_avg = regional.groupby('region')['recycling_rate'].mean()
best_state  = state_avg.idxmax()
worst_state = state_avg.idxmin()
print(f'''4. REGIONAL DIVIDE
   Best performing state  : {best_state} ({state_avg[best_state]:.1f}% avg across all materials)
   Worst performing state : {worst_state} ({state_avg[worst_state]:.1f}% avg across all materials)
   Gap = {state_avg[best_state] - state_avg[worst_state]:.1f} percentage points between best and worst.
''')

# Finding 5: Global
us_global = latest_global[latest_global['country']=='United States']['recycling_rate'].values[0]
top_c     = latest_global.iloc[-1]
print(f'''5. GLOBAL GAP
   US recycling rate (MSW overall) : {us_global}%
   Top country: {top_c["country"]} at {top_c["recycling_rate"]}%
   The US recycles roughly half as much as the top performers.
   Strong correlation between national recycling policy strength and rate.
''')

print('=' * 65)

In [ ]:
# Final summary chart — one image with all key numbers
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1: 2018 rates
rates_2018 = national[national['year']==2018].set_index('material')['recycling_rate']
bars = axes[0].bar(rates_2018.index, rates_2018.values,
                   color=[COLORS[m] for m in rates_2018.index],
                   alpha=0.85, edgecolor='white', width=0.6)
for bar, val in zip(bars, rates_2018.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('US Rates in 2018', fontweight='bold')
axes[0].set_ylabel('Recycling Rate (%)')
axes[0].set_ylim(0, 85)

# 2: State avg heatmap (simplified)
state_mat = regional.pivot_table(index='region', columns='material', values='recycling_rate')
state_mat = state_mat.loc[state_mat.mean(axis=1).sort_values(ascending=False).index]
sns.heatmap(state_mat[['Plastic','Glass','Metal','Paper']], annot=True, fmt='.0f',
            cmap='YlGn', linewidths=0.3, linecolor='white',
            cbar=False, ax=axes[1], annot_kws={'size': 9})
axes[1].set_title('State × Material Rates (%)', fontweight='bold')
axes[1].set_xlabel('')

# 3: Global top 8
top8 = latest_global.tail(8)
colors_g = ['#ef5350' if c=='United States' else '#2e7d32' for c in top8['country']]
axes[2].barh(top8['country'], top8['recycling_rate'], color=colors_g, alpha=0.85)
axes[2].set_title('Global Leaders (MSW)', fontweight='bold')
axes[2].set_xlabel('Rate (%)')
axes[2].set_xlim(0, 65)

plt.suptitle('Recycling Dashboard — Summary of Key Findings', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('chart_summary.png', bbox_inches='tight', dpi=150)
plt.show()
print('\nAll charts saved! Use these PNG files in your final report.')
print('Files: chart_national_trends, chart_distributions, chart_boxplot,')
print('       chart_regression, chart_regional, chart_heatmap,')
print('       chart_deposit_law, chart_global, chart_correlation, chart_summary')